# Notebook 03: SageMaker Batch Transform Job Dự Đoán Hàng Loạt (Tuần 3)
**Môn học:** Cloud - Đợt 1 - 2026-2027  
**Đề tài:** Phân loại bệnh lá cây bằng Transfer Learning trên SageMaker  
**Mô hình sử dụng:** MobileNetV2 Artifact (`model.tar.gz` đã huấn luyện ở Tuần 2)  
**Compute Instance:** `ml.c5.xlarge` (CPU)  
**Ưu điểm chi phí:** Máy ảo tự động bật lên xử lý dự đoán toàn bộ tập ảnh và **tự động giải phóng ngay khi hoàn tất**, không phát sinh chi phí duy trì.

In [ ]:
import os
import json
import boto3
import sagemaker
from sagemaker.pytorch.model import PyTorchModel

# 1. Khởi tạo cấu hình môi trường
session = sagemaker.Session()
account_id = boto3.client('sts').get_caller_identity()['Account']
role = f"arn:aws:iam::{account_id}:role/LabRole"
bucket_name = f"sagemaker-plant-leaf-{account_id}"

# Model Artifact MobileNetV2 đã huấn luyện ở Tuần 2
model_data = f"s3://{bucket_name}/output/pytorch-training-2026-10-18-16-35-53-425/output/model.tar.gz"

# Kênh dữ liệu kiểm thử và kênh xuất kết quả
s3_test_data = f"s3://{bucket_name}/dataset/val"
s3_transform_output = f"s3://{bucket_name}/batch_transform_output"

print(f"Account ID     : {account_id}")
print(f"IAM Role       : {role}")
print(f"Model Artifact : {model_data}")
print(f"Dữ liệu đầu vào: {s3_test_data}")
print(f"Nơi xuất kết quả: {s3_transform_output}")

In [ ]:
# 2. Xác định thư mục mã nguồn inference.py
source_candidates = ['Cloud/src', '../src', 'src']
source_dir = next((d for d in source_candidates if os.path.exists(d)), '../src')
print(f"Thư mục chứa inference.py: {source_dir}")

# Khởi tạo PyTorchModel với custom handler inference.py
pytorch_model = PyTorchModel(
    model_data=model_data,
    role=role,
    entry_point='inference.py',
    source_dir=source_dir,
    framework_version='2.1.0',
    py_version='py310',
    sagemaker_session=session
)
print("[OK] PyTorchModel đã được cấu hình sẵn sàng.")

In [ ]:
# 3. Khởi tạo Transformer và kích hoạt Batch Transform Job
transformer = pytorch_model.transformer(
    instance_count=1,
    instance_type='ml.c5.xlarge',      # Chạy trên CPU instance tiết kiệm chi phí
    output_path=s3_transform_output,
    accept='application/json',
    max_payload=6
)

print("Đang submit Batch Transform Job lên AWS SageMaker...")
# Chạy suy luận hàng loạt trên tập ảnh
transformer.transform(
    data=s3_test_data,
    content_type='application/x-image',
    split_type='None'
)

# Chờ hoàn tất và hiển thị log thời gian thực
transformer.wait()

In [ ]:
# 4. Đọc mẫu kết quả suy luận từ S3 để kiểm chứng
s3 = boto3.client('s3')
prefix = "batch_transform_output/"
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix, MaxKeys=5)

print("=" * 60)
print("MẪU KẾT QUẢ DỰ ĐOÁN TỪ BATCH TRANSFORM JOB:")
print("=" * 60)

if 'Contents' in response:
    for obj in response['Contents'][:5]:
        key = obj['Key']
        if key.endswith('.out'):
            data = s3.get_object(Bucket=bucket_name, Key=key)['Body'].read().decode('utf-8')
            print(f"File: {key}")
            print(f"Kết quả: {data}\n")
else:
    print("Chưa tìm thấy file kết quả .out trên S3. Vui lòng kiểm tra lại prefix.")

print("[THÀNH CÔNG] Batch Transform hoàn tất. Máy ảo đã tự động giải phóng!")